# 🧠 Agent Memory

**Give your agents short-term and long-term memory**

---

## 📋 Overview

**What you'll learn:**
- Short-term vs long-term memory
- Conversation memory
- Entity memory
- Vector memory (semantic)
- Memory management strategies

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from openai import OpenAI
import os
import json
from typing import Dict, List, Any
from datetime import datetime

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 Why Memory?

### Without Memory:
```
User: "My name is Alice"
Agent: "Nice to meet you!"

User: "What's my name?"
Agent: "I don't know your name."
❌ Agent forgot!
```

### With Memory:
```
User: "My name is Alice"
Agent: "Nice to meet you, Alice!" [stores: name=Alice]

User: "What's my name?"
Agent: "Your name is Alice" [retrieves from memory]
✅ Agent remembers!
```

### Types of Memory:

**1. Short-Term (Working Memory)**
- Current conversation
- Last few messages
- Temporary context

**2. Long-Term Memory**
- User preferences
- Past conversations
- Learned facts

**3. Entity Memory**
- People: "Alice, Bob"
- Places: "Paris, London"
- Things: "MacBook, iPhone"

**4. Semantic Memory**
- Vector embeddings
- Similar past interactions
- RAG-style retrieval

## 💬 Conversation Memory

In [ ]:
class ConversationMemory:
    """Simple conversation memory."""
    
    def __init__(self, max_messages: int = 10):
        self.messages = []
        self.max_messages = max_messages
    
    def add_message(self, role: str, content: str):
        """Add a message to memory."""
        self.messages.append({
            "role": role,
            "content": content,
            "timestamp": datetime.now().isoformat()
        })
        
        # Trim if too long (keep system + recent messages)
        if len(self.messages) > self.max_messages:
            system_msgs = [m for m in self.messages if m['role'] == 'system']
            recent_msgs = self.messages[-(self.max_messages-len(system_msgs)):]
            self.messages = system_msgs + recent_msgs
    
    def get_messages(self) -> List[Dict]:
        """Get all messages for LLM."""
        return [{"role": m["role"], "content": m["content"]} for m in self.messages]
    
    def summarize(self) -> str:
        """Summarize conversation."""
        if not self.messages:
            return "No conversation yet"
        
        return f"{len(self.messages)} messages exchanged"

# Example usage
memory = ConversationMemory(max_messages=5)

memory.add_message("system", "You are a helpful assistant")
memory.add_message("user", "My name is Alice")
memory.add_message("assistant", "Nice to meet you, Alice!")
memory.add_message("user", "I like Python")
memory.add_message("assistant", "Python is a great language!")
memory.add_message("user", "What's my name?")

print("💬 Conversation Memory\n")
print("Messages in memory:")
for msg in memory.get_messages():
    print(f"  {msg['role']}: {msg['content']}")

# Agent can now answer "What's my name?"
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=memory.get_messages()
)

print(f"\n🤖 Agent: {response.choices[0].message.content}")

## 👤 Entity Memory

In [ ]:
class EntityMemory:
    """Remember entities (people, places, things)."""
    
    def __init__(self):
        self.entities = {}
    
    def extract_and_store(self, text: str):
        """Extract entities from text using LLM."""
        
        extraction_prompt = f"""Extract key entities from this text.

Text: {text}

Extract:
- Names (people)
- Preferences (likes, dislikes)
- Facts (important info)

Return as JSON:
{{
  "name": "...",
  "preferences": [...],
  "facts": [...]
}}

JSON:"""
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": extraction_prompt}],
            temperature=0
        )
        
        try:
            entities = json.loads(response.choices[0].message.content)
            
            # Store entities
            if entities.get("name"):
                self.entities["name"] = entities["name"]
            
            if entities.get("preferences"):
                self.entities.setdefault("preferences", [])
                self.entities["preferences"].extend(entities["preferences"])
            
            if entities.get("facts"):
                self.entities.setdefault("facts", [])
                self.entities["facts"].extend(entities["facts"])
            
            return entities
        except:
            return {}
    
    def get_context(self) -> str:
        """Get entity context for LLM."""
        if not self.entities:
            return "No stored information about the user."
        
        context = "Known information:\n"
        
        if "name" in self.entities:
            context += f"- Name: {self.entities['name']}\n"
        
        if "preferences" in self.entities:
            context += f"- Preferences: {', '.join(self.entities['preferences'])}\n"
        
        if "facts" in self.entities:
            context += f"- Facts: {', '.join(self.entities['facts'])}\n"
        
        return context

# Example
entity_memory = EntityMemory()

print("👤 Entity Memory Example\n")

# User shares information
user_statements = [
    "My name is Bob",
    "I love Python and machine learning",
    "I work at TechCorp as a software engineer"
]

for statement in user_statements:
    print(f"User: {statement}")
    entities = entity_memory.extract_and_store(statement)
    print(f"  Extracted: {entities}\n")

print("\n📝 Stored Entity Memory:")
print(entity_memory.get_context())

# Agent can now use this context
print("\n💡 Agent can now personalize responses using stored entities!")

## 🔍 Semantic Memory (Vector-Based)

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

class SemanticMemory:
    """Vector-based semantic memory."""
    
    def __init__(self):
        self.client = chromadb.Client()
        
        # Use OpenAI embeddings
        openai_ef = embedding_functions.OpenAIEmbeddingFunction(
            api_key=os.getenv('OPENAI_API_KEY'),
            model_name="text-embedding-3-small"
        )
        
        self.collection = self.client.create_collection(
            name="agent_memory",
            embedding_function=openai_ef
        )
    
    def remember(self, text: str, metadata: Dict = None):
        """Store a memory."""
        memory_id = f"memory_{datetime.now().timestamp()}"
        
        self.collection.add(
            documents=[text],
            metadatas=[metadata or {}],
            ids=[memory_id]
        )
        
        print(f"✅ Stored: {text[:50]}...")
    
    def recall(self, query: str, n_results: int = 3) -> List[str]:
        """Recall relevant memories."""
        
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results
        )
        
        if results['documents']:
            return results['documents'][0]
        return []

# Example
semantic_memory = SemanticMemory()

print("🔍 Semantic Memory Example\n")
print("Storing memories...\n")

# Store various memories
memories = [
    "User prefers Python for data science projects",
    "User mentioned they work at TechCorp",
    "User likes to use Jupyter notebooks for experiments",
    "User asked about machine learning best practices",
    "User is interested in deep learning frameworks"
]

for memory in memories:
    semantic_memory.remember(memory)

# Recall relevant memories
query = "What programming language does the user like?"
print(f"\n🔍 Query: {query}")

relevant_memories = semantic_memory.recall(query)
print("\n📝 Relevant memories:")
for i, mem in enumerate(relevant_memories, 1):
    print(f"  {i}. {mem}")

print("\n💡 Agent can use these memories to provide personalized responses!")

## 🧠 Complete Agent with Memory

In [ ]:
class AgentWithMemory:
    """Agent with comprehensive memory system."""
    
    def __init__(self):
        self.conversation_memory = ConversationMemory(max_messages=10)
        self.entity_memory = EntityMemory()
        # self.semantic_memory = SemanticMemory()  # Optional
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    
    def chat(self, user_message: str) -> str:
        """Chat with memory."""
        
        # Extract entities from user message
        self.entity_memory.extract_and_store(user_message)
        
        # Add user message to conversation memory
        self.conversation_memory.add_message("user", user_message)
        
        # Build context with entity memory
        entity_context = self.entity_memory.get_context()
        
        # Prepend entity context to messages
        messages = [
            {
                "role": "system",
                "content": f"""You are a helpful assistant with memory.

{entity_context}

Use this information to provide personalized responses."""
            }
        ] + self.conversation_memory.get_messages()
        
        # Get response
        response = self.client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages
        )
        
        assistant_message = response.choices[0].message.content
        
        # Add to conversation memory
        self.conversation_memory.add_message("assistant", assistant_message)
        
        return assistant_message

# Example conversation
agent = AgentWithMemory()

print("🧠 Agent with Complete Memory\n")
print("="*60)

conversation = [
    "Hi, my name is Charlie",
    "I'm learning about AI agents",
    "What should I learn next?",
    "What's my name again?"
]

for user_msg in conversation:
    print(f"\n👤 User: {user_msg}")
    response = agent.chat(user_msg)
    print(f"🤖 Agent: {response}")

print("\n" + "="*60)
print("\n✅ Agent remembered the name and context throughout conversation!")

## ✅ Summary

### Memory Types:

**1. Conversation Memory**
```python
# Short-term: Last N messages
messages = [
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
]
```

**2. Entity Memory**
```python
# Structured facts
entities = {
    "name": "Alice",
    "preferences": ["Python", "ML"],
    "facts": ["Works at TechCorp"]
}
```

**3. Semantic Memory**
```python
# Vector-based retrieval
relevant_memories = vector_db.query("user preferences")
```

### Memory Management:

**Window Strategy:**
```python
# Keep only recent N messages
if len(messages) > 10:
    messages = messages[-10:]
```

**Summarization Strategy:**
```python
# Summarize old messages
if len(messages) > 20:
    summary = summarize(messages[:10])
    messages = [summary] + messages[10:]
```

**Importance-Based:**
```python
# Keep important messages
important = [m for m in messages if m['importance'] > 0.8]
recent = messages[-5:]
messages = important + recent
```

### Best Practices:

**1. Layered Memory**
```python
# Use multiple memory types
- Conversation (immediate context)
- Entity (key facts)
- Semantic (historical relevance)
```

**2. Limit Context**
```python
# Don't exceed token limits
max_tokens = 4000  # For GPT-3.5
context = truncate_to_tokens(memory, max_tokens)
```

**3. Relevance Filtering**
```python
# Only include relevant memories
relevant = semantic_memory.query(current_topic, top_k=3)
```

**4. Persistence**
```python
# Save memory to database
db.save(user_id, memory)

# Load on next session
memory = db.load(user_id)
```

### Memory Architecture:

```
User Input
    ↓
┌───────────────────┐
│ Conversation      │ ← Immediate context
│ Memory (short)    │
└───────────────────┘
    ↓
┌───────────────────┐
│ Entity Memory     │ ← Extracted facts
│ (structured)      │
└───────────────────┘
    ↓
┌───────────────────┐
│ Semantic Memory   │ ← Historical context
│ (vector DB)       │
└───────────────────┘
    ↓
Combined Context → LLM
```

### Next: `07_agents_tools/06_multi_agent.ipynb`